# MARKET PULSE — 실제 국내 주가 데이터 분석

이 노트북에는 **생성된 가상 주가·뉴스가 포함되지 않습니다.** 원본 두 ZIP은 API 소스 및 참고 자료이며 일별 시세 레코드를 포함하지 않습니다. 인터넷 연결로 FinanceDataReader에서 실거래일 OHLCV를 받아 분석합니다.

실행 방법: GitHub의 프로젝트 ZIP을 Colab에 업로드해 압축 해제한 뒤 순서대로 실행하거나, Colab에서 프로젝트를 체크아웃하고 루트로 이동합니다.

In [ ]:
%pip -q install 'finance-datareader>=0.9.90' 'pandas>=2,<3' 'numpy>=1.24,<3' 'plotly>=5.18,<7' 'scikit-learn>=1.3,<2' 'requests>=2.31,<3'

## 1. 프로젝트 파일 준비
아래 셀은 ZIP을 업로드하고 풀어 상대 경로에서 `stock_core.py`, `market_data.py` 등 실제 모듈이 발견되는지 확인합니다. 사용자가 GitHub 경로를 직접 준비한 경우 ZIP 업로드 부분을 생략해도 됩니다.

In [ ]:
from pathlib import Path
import os, zipfile
from google.colab import files
print('MARKET PULSE 프로젝트 ZIP 파일을 업로드하세요 (FinanceDataReader-master.zip 아님).')
uploaded = files.upload()
name = next((k for k in uploaded if k.lower().endswith('.zip')), None)
if not name:
    raise RuntimeError('MARKET PULSE 전체 프로젝트 ZIP을 업로드해야 합니다.')
with zipfile.ZipFile(name) as z:
    paths=[Path(x) for x in z.namelist()]
    if not any(x.name == 'stock_core.py' for x in paths):
        raise RuntimeError('올린 ZIP에 MARKET PULSE stock_core.py 파일이 없습니다.')
    # Extract only paths without traversal for safety.
    for item in z.infolist():
        pp = Path(item.filename)
        if pp.is_absolute() or '..' in pp.parts:
            raise RuntimeError('안전하지 않은 ZIP 경로 발견')
    z.extractall('/content/mp_workspace')
project_files=list(Path('/content/mp_workspace').rglob('stock_core.py'))
project_root = project_files[0].parent
os.chdir(project_root)
print('프로젝트 폴더:',project_root)

## 2. 실제 주가 조회 및 출처 기록
다음 코드는 KRX 종목코드로 실제 시세를 요청합니다. 조회 실패 시 캐시가 없는 경우 **오류를 표시하고 멈춥니다.** API 기록이 없는 날짜를 임의로 채우지 않습니다.

In [ ]:
from datetime import date
from market_data import fetch_historical
from stock_core import calculate_indicators, trim_period, summarize, event_table
from technical_indicators import add_indicators

SYMBOL = '005930'  # 삼성전자 종목코드 (식별자일 뿐, 가격이 아님)
START = date(2024,1,1)
END = date.today()
result = fetch_historical(SYMBOL, START, END)
if result.meta.get('status') != 'fresh':
    print('주의: 최신 수집 실패. 저장된 실제 출처 캐시입니다.')
print('실제 데이터 제공처:',result.meta.get('source'))
print('수집 시각:',result.meta.get('retrieved_at'))
print('마지막 거래일:',result.meta.get('last_trade_date'))
prices = result.frame
display(prices.tail(10))

## 3. 통계 분석과 기술적 지표

In [ ]:
df=trim_period(calculate_indicators(prices), START, END)
indicators=add_indicators(prices)
display(summarize({SYMBOL: df}))
display(event_table({SYMBOL: df}).head(20))
display(indicators[['Date','Close','Volume','SMA_20','RSI_14','MACD']].tail(20))

## 4. 머신러닝 예측 연구(실제 과거 시세만 입력)
학습 결과는 미래 가격 보장이 아니며 뉴스 데이터가 없는 경우 뉴스 결합 모델이 생성되지 않습니다.

In [ ]:
from prediction_model import run_experiments
experiment=run_experiments(prices)
display(experiment.summary)
display(experiment.forecasts)
print('뉴스 데이터가 없는 상태:',experiment.news_note)

## 5. 검증 데이터 다운로드
출처 및 거래일 메타데이터를 함께 기록합니다. 생성된 결과는 사용자가 온라인에서 실제 조회한 데이터에 기반합니다.

In [ ]:
from pathlib import Path
import pandas as pd, json
output=Path('colab_real_outputs');output.mkdir(exist_ok=True)
for filename, table in {
   'ohlcv.csv':prices,
   'anomalies.csv':event_table({SYMBOL:df}),
   'model_scores.csv':experiment.summary,
   'forecasts.csv':experiment.forecasts,
}.items():
    table.to_csv(output/filename,index=False,encoding='utf-8-sig')
(output/'source_metadata.json').write_text(json.dumps(result.meta,ensure_ascii=False,indent=2),encoding='utf-8')
print('출력 폴더:',output)